In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.4 MB/s eta 0:00:00


In [ ]:
#first create the ultimate dataset with all the wanted columns for the rest of the pipeline
import pandas as pd

# Load existing blast summary
input_path = "/home/k_ensafitakaldani001_umb_edu/BLAST/target_malaria_blast_results.csv"
df = pd.read_csv(input_path)

# Rename existing columns
df = df.rename(columns={
    'target_name': 'target',
    'malaria_name': 'malaria',
    'target_id': 'T-cid',
    'malaria_id': 'M-cid',
    'e_value': 'blast_evalue',
    'bit_score': 'blast_bit_score',
    'percent_identity': 'blast_percent_identity',
    'alignment_length': 'blast_alignment_length'
})

# Define the final column schema
final_columns = [
    'target', 'malaria', 'target_renamed', 'malaria_renamed', 'T-cid', 'M-cid', 'T-lig', 'M-lig', 'T-gene', 'M-gene',
    'pubchem_cid', 'smiles', 'blast_evalue', 'blast_alignment_length','blast_bit_score', 'blast_percent_identity',
    'ligand_D', 'M_D_align_evalue', 'M_D_rmsd', 'M_D_align_len','3d_align_evalue', '3d_rmsd', '3d_align_len'
]

# Add missing columns with NaN
for col in final_columns:
    if col not in df.columns:
        df[col] = pd.NA

# Reorder the columns
df = df[final_columns]

#add the renamed target and malaria -> just copy the name and concat _renamed.pdb at the end which is for the alignment and its also the name of the renamed files with T D M
df['target_renamed'] = df['target'].astype(str) + "_renamed.pdb"
df['malaria_renamed'] = df['malaria'].astype(str) + "_renamed.pdb"

# Save the updated dataset
df.to_csv(input_path, index=False)
print("T-rename and M-rename columns updated and saved.")

# Save the ultimate dataset
output_path = "/home/k_ensafitakaldani001_umb_edu/BLAST/ultimate_target_malaria_dataset.csv"
df.to_csv(output_path, index=False)

print(f"Ultimate dataset saved to: {output_path}")


In [ ]:
df = pd.read_csv('/home/k_ensafitakaldani001_umb_edu/BLAST/ultimate_target_malaria_dataset.csv')
df.head(1)

In [2]:
from Bio.PDB import PDBParser, PPBuilder

def parse_chains_and_ligands_with_atom_sequences(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', pdb_file)
    ppb = PPBuilder()

    results = {}

    for model in structure:
        for chain in model:
            chain_id = chain.id

            # Extract protein sequence
            seq = ''
            for pp in ppb.build_peptides(chain):
                seq += str(pp.get_sequence())

            # Extract ligands connected to this chain with their "atom sequence"
            ligands = []
            for residue in chain:
                hetfield, resseq, icode = residue.get_id()
                if hetfield != ' ' and residue.get_resname() != 'HOH':
                    atom_sequence = []
                    for atom in residue:
                        atom_sequence.append({
                            'atom_name': atom.get_name(),
                            'element': atom.element
                        })
                    ligands.append({
                        'residue_name': residue.get_resname(),
                        'resseq': resseq,
                        'insertion_code': icode,
                        'atom_sequence': atom_sequence
                    })

            results[chain_id] = {
                'protein_sequence': seq,
                'ligands': ligands
            }

    return results


# Example usage

!wget -qnc https://files.rcsb.org/download/1HHO.pdb

pdb_file = '/content/1HHO.pdb'  # replace with your file path
data = parse_chains_and_ligands_with_atom_sequences(pdb_file)

# Print results
for chain_id, content in data.items():
    print(f"=== Chain {chain_id} ===")
    print("Protein Sequence:")
    print(content['protein_sequence'])
    print("Ligands:")
    if content['ligands']:
        for lig in content['ligands']:
            print(f"  Ligand {lig['residue_name']} {lig['resseq']}{lig['insertion_code']}:")
            print("    Atom Sequence:")
            atom_seq_str = ' - '.join([f"{a['atom_name']}({a['element']})" for a in lig['atom_sequence']])
            print(f"    {atom_seq_str}")
    else:
        print("  None")
    print()


=== Chain A ===
Protein Sequence:
VLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR
Ligands:
  Ligand PO4 142 :
    Atom Sequence:
    P(P) - O1(O) - O2(O) - O3(O) - O4(O)
  Ligand HEM 143 :
    Atom Sequence:
    CHA(C) - CHB(C) - CHC(C) - CHD(C) - C1A(C) - C2A(C) - C3A(C) - C4A(C) - CMA(C) - CAA(C) - CBA(C) - CGA(C) - O1A(O) - O2A(O) - C1B(C) - C2B(C) - C3B(C) - C4B(C) - CMB(C) - CAB(C) - CBB(C) - C1C(C) - C2C(C) - C3C(C) - C4C(C) - CMC(C) - CAC(C) - CBC(C) - C1D(C) - C2D(C) - C3D(C) - C4D(C) - CMD(C) - CAD(C) - CBD(C) - CGD(C) - O1D(O) - O2D(O) - NA(N) - NB(N) - NC(N) - ND(N) - FE(FE)
  Ligand OXY 150 :
    Atom Sequence:
    O1(O) - O2(O)

=== Chain B ===
Protein Sequence:
VHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH
Ligands:
  Ligand HEM 147 :
    Atom Sequence:
    CHA(C) - CHB(C) - CHC(C)

In [3]:
from Bio.PDB import PDBParser, NeighborSearch, Selection, PPBuilder

def find_ligand_binding_sites(pdb_file, distance_cutoff=5.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', pdb_file)

    model = next(structure.get_models())  # get first model

    # Build list of all protein atoms for NeighborSearch
    atoms = Selection.unfold_entities(model, 'A')  # 'A' for atoms
    ns = NeighborSearch(atoms)

    results = []

    for chain in model:
        chain_id = chain.id

        # Extract protein sequence
        ppb = PPBuilder()
        seq = ''
        for pp in ppb.build_peptides(chain):
            seq += str(pp.get_sequence())

        # Find ligands in this chain
        for residue in chain:
            hetfield, resseq, icode = residue.get_id()
            if hetfield != ' ' and residue.get_resname() != 'HOH':
                ligand_atoms = list(residue.get_atoms())
                ligand_name = residue.get_resname()

                # Find nearby residues (binding site)
                binding_residues = set()
                for atom in ligand_atoms:
                    neighbors = ns.search(atom.get_coord(), distance_cutoff, level='R')
                    for neighbor in neighbors:
                        if neighbor.get_parent().id == chain_id:
                            # Only include protein residues, skip heteroatoms
                            n_hetfield = neighbor.get_id()[0]
                            if n_hetfield == ' ':
                                res_id = neighbor.get_id()[1]
                                res_name = neighbor.get_resname()
                                binding_residues.add( (res_id, res_name) )

                results.append({
                    'chain_id': chain_id,
                    'ligand_name': ligand_name,
                    'ligand_resseq': resseq,
                    'binding_site_residues': sorted(binding_residues)
                })

    return results

# Example usage
pdb_file = '/content/1HHO.pdb'  # replace with your file path
binding_sites = find_ligand_binding_sites(pdb_file)

# Print results
for entry in binding_sites:
    print(f"Chain {entry['chain_id']} - Ligand {entry['ligand_name']} {entry['ligand_resseq']}")
    print("Binding site residues:")
    for res in entry['binding_site_residues']:
        print(f"  Residue {res[1]} {res[0]}")
    print()


Chain A - Ligand PO4 142
Binding site residues:
  Residue LYS 99
  Residue ARG 141

Chain A - Ligand HEM 143
Binding site residues:
  Residue MET 32
  Residue THR 39
  Residue TYR 42
  Residue PHE 43
  Residue HIS 45
  Residue PHE 46
  Residue HIS 58
  Residue LYS 61
  Residue VAL 62
  Residue ALA 65
  Residue LEU 66
  Residue LEU 83
  Residue LEU 86
  Residue HIS 87
  Residue LEU 91
  Residue VAL 93
  Residue ASN 97
  Residue PHE 98
  Residue LEU 101
  Residue SER 102
  Residue LEU 105
  Residue LEU 129
  Residue VAL 132
  Residue SER 133
  Residue LEU 136

Chain A - Ligand OXY 150
Binding site residues:
  Residue LEU 29
  Residue PHE 43
  Residue HIS 58
  Residue VAL 62
  Residue HIS 87
  Residue LEU 101

Chain B - Ligand HEM 147
Binding site residues:
  Residue LEU 31
  Residue THR 38
  Residue PHE 41
  Residue PHE 42
  Residue HIS 63
  Residue LYS 66
  Residue VAL 67
  Residue ALA 70
  Residue PHE 71
  Residue PHE 85
  Residue LEU 88
  Residue LEU 91
  Residue HIS 92
  Residue LEU 

In [4]:
def fetch_json(url):
    r = requests.get(url)
    if r.status_code != 200:
        raise Exception(f"Failed to fetch {url} — {r.status_code}")
    return r.json()

def get_structure_summary_df(pdb_id):
    entry_url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    entry_data = fetch_json(entry_url)

    polymer_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    ligand_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("non_polymer_entity_ids", [])

    # Full extraction (even if only some is returned)
    molecule_list = []
    chains_list = []
    gene_names_list = []
    organisms_list = []
    lengths_list = []
    mutations_list = []

    for entity_id in polymer_ids:
        poly_url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}"
        poly = fetch_json(poly_url)

        molecule = poly.get("entity", {}).get("pdbx_description", "N/A")
        chains = poly.get("rcsb_polymer_entity", {}).get("pdbx_strand_id", [])
        orgs = [o.get("ncbi_scientific_name", "N/A") for o in poly.get("rcsb_entity_source_organism", [])]
        gene_names = [g.get("value") for g in poly.get("rcsb_entity_source_organism", [{}])[0].get("rcsb_gene_name", [])] if poly.get("rcsb_entity_source_organism") else []
        length = poly.get("entity_poly", {}).get("rcsb_sample_sequence_length", "N/A")
        mutations = poly.get("entity_poly", {}).get("rcsb_mutation_count", "N/A")

        molecule_list.append(molecule)
        chains_list.append(", ".join(chains))
        gene_names_list.append(", ".join(gene_names) if gene_names else "N/A")
        organisms_list.append(", ".join(orgs))
        lengths_list.append(length)
        mutations_list.append(mutations)


    ligand_names = []
    for entity_id in ligand_ids:
        lig_url = f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{entity_id}"
        lig = fetch_json(lig_url)
        nonpoly = lig.get("pdbx_entity_nonpoly", {})
        name = nonpoly.get("name", "N/A")
        comp_id = nonpoly.get("comp_id", "N/A")
        ligand_names.append(f"{comp_id}: {name}")

    return pd.DataFrame([{
        "PDB_ID": pdb_id,
        "Molecules": molecule_list,
        "Chains": chains_list,
        "Gene_Names": gene_names_list,
        "Organisms": organisms_list,
        "Sequence_Lengths": lengths_list,
        "Mutations": mutations_list,
        "Ligands": ligand_names
    }])


In [5]:
import pandas as pd
csvpath = '/content/ultimate_target_malaria_dataset.csv'
df2 = pd.read_csv(csvpath)
df2.head()

,target,malaria,target_renamed,malaria_renamed,T-cid,M-cid,T-lig,M-lig,T-gene,M-gene,...,blast_alignment_length,blast_bit_score,blast_percent_identity,ligand_D,M_D_align_evalue,M_D_rmsd,M_D_align_len,3d_align_evalue,3d_rmsd,3d_align_len
0,1C14,3AM3,1C14_renamed.pdb,3AM3_renamed.pdb,A,A,NaN,NaN,NaN,NaN,...,237,87.0,25.738,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1C14,1UH5,1C14_renamed.pdb,1UH5_renamed.pdb,A,A,NaN,NaN,NaN,NaN,...,237,85.5,25.316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1C14,2OL4,1C14_renamed.pdb,2OL4_renamed.pdb,A,A,NaN,NaN,NaN,NaN,...,237,84.3,24.051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1C14,1VRW,1C14_renamed.pdb,1VRW_renamed.pdb,A,A,NaN,NaN,NaN,NaN,...,237,84.0,24.051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1C14,3AM5,1C14_renamed.pdb,3AM5_renamed.pdb,A,A,NaN,NaN,NaN,NaN,...,237,83.6,25.316,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
import pandas as pd
import requests

#FOR THE TARGETS
# Step 1: Create lookup table from df2['target_chain_id']
df_list = []


for pdb in df2['target']:
    try:
      #drop chain ID
        pdb = pdb[:4]
        df = get_structure_summary_df(pdb)
        df_list.append(df)
    except Exception as e:
        print(f"Error processing {pdb}: {e}")

# Step 2: Combine all fetched info
combined_df = pd.concat(df_list, ignore_index=True)
lookup = {
    row["PDB_ID"]: {
        "Gene_Names": row["Gene_Names"],
        "Ligands": row["Ligands"]
    }
    for _, row in combined_df.iterrows()
}

# Step 3: Add to original df2
df2["T-gene"] = df2["target"].str[:4].apply(lambda x: lookup.get(x, {}).get("Gene_Names", ["N/A"]))
df2["T-lig"] = df2["target"].str[:4].apply(lambda x: lookup.get(x, {}).get("Ligands", ["N/A"]))


In [8]:
df2.head(10)

,target,malaria,target_renamed,malaria_renamed,T-cid,M-cid,T-lig,M-lig,T-gene,M-gene,...,blast_alignment_length,blast_bit_score,blast_percent_identity,ligand_D,M_D_align_evalue,M_D_rmsd,M_D_align_len,3d_align_evalue,3d_rmsd,3d_align_len
0,1C14,3AM3,1C14_renamed.pdb,3AM3_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,237,87.0,25.738,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1C14,1UH5,1C14_renamed.pdb,1UH5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,237,85.5,25.316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1C14,2OL4,1C14_renamed.pdb,2OL4_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,237,84.3,24.051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1C14,1VRW,1C14_renamed.pdb,1VRW_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,237,84.0,24.051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1C14,3AM5,1C14_renamed.pdb,3AM5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,237,83.6,25.316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1C14,1NHG,1C14_renamed.pdb,1NHG_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,141,70.1,29.078,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1D7O,3AM3,1D7O_renamed.pdb,3AM3_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,321,301.0,47.975,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1D7O,3AM5,1D7O_renamed.pdb,3AM5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,321,298.0,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1D7O,1UH5,1D7O_renamed.pdb,1UH5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,321,298.0,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1D7O,2OL4,1D7O_renamed.pdb,2OL4_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",NaN,[N/A],NaN,...,321,298.0,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
#FOR THE MALARIA

# Step 1: Create lookup table from df2['target_chain_id']
df_list = []

for pdb in df2['malaria']:
    try:
        pdb = pdb[:4]
        df = get_structure_summary_df(pdb)
        df_list.append(df)
    except Exception as e:
        print(f"Error processing {pdb}: {e}")

# Step 2: Combine all fetched info
combined_df = pd.concat(df_list, ignore_index=True)
lookup = {
    row["PDB_ID"]: {
        "Gene_Names": row["Gene_Names"],
        "Ligands": row["Ligands"]
    }
    for _, row in combined_df.iterrows()
}

# Step 3: Add to original df2
df2["M-gene"] = df2["malaria"].str[:4].apply(lambda x: lookup.get(x, {}).get("Gene_Names", ["N/A"]))
df2["M-lig"] = df2["malaria"].str[:4].apply(lambda x: lookup.get(x, {}).get("Ligands", ["N/A"]))


In [12]:
df2.head(10)

,target,malaria,target_renamed,malaria_renamed,T-cid,M-cid,T-lig,M-lig,T-gene,M-gene,...,blast_alignment_length,blast_bit_score,blast_percent_identity,ligand_D,M_D_align_evalue,M_D_rmsd,M_D_align_len,3d_align_evalue,3d_rmsd,3d_align_len
0,1C14,3AM3,1C14_renamed.pdb,3AM3_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[N/A],[FabI],...,237,87.0,25.738,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1C14,1UH5,1C14_renamed.pdb,1UH5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[TCL: TRICLOSAN, NAD: NICOTINAMIDE-ADENINE-DIN...",[N/A],[FabI],...,237,85.5,25.316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1C14,2OL4,1C14_renamed.pdb,2OL4_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, JPN: ...",[N/A],[FabI],...,237,84.3,24.051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1C14,1VRW,1C14_renamed.pdb,1VRW_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAI: 1,4-DIHYDRONICOTINAMIDE ADENINE DINUCLEO...",[N/A],[N/A],...,237,84.0,24.051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1C14,3AM5,1C14_renamed.pdb,3AM5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[N/A],[fabI],...,237,83.6,25.316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1C14,1NHG,1C14_renamed.pdb,1NHG_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[N/A],"[N/A, N/A]",...,141,70.1,29.078,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1D7O,3AM3,1D7O_renamed.pdb,3AM3_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[N/A],[FabI],...,321,301.0,47.975,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1D7O,3AM5,1D7O_renamed.pdb,3AM5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",[N/A],[fabI],...,321,298.0,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1D7O,1UH5,1D7O_renamed.pdb,1UH5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[TCL: TRICLOSAN, NAD: NICOTINAMIDE-ADENINE-DIN...",[N/A],[FabI],...,321,298.0,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1D7O,2OL4,1D7O_renamed.pdb,2OL4_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, JPN: ...",[N/A],[FabI],...,321,298.0,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df2.to_csv('/content/Ultimate_Dataset.csv', index = False)

In [16]:
import pandas as pd
import ast

csvpath = '/content/target_malaria_ligand_gene.csv'
df = pd.read_csv(csvpath)

#TARGET LIGANDs
df['T-lig'] = df['T-lig'].apply(ast.literal_eval)
# Extract only the ligand codes (before colon) into a new column
df['T-lig-code'] = df['T-lig'].apply(lambda lst: [item.split(':')[0].strip() for item in lst])

#MALARIA LIGANDs
df['M-lig'] = df['M-lig'].apply(ast.literal_eval)
# Extract only the ligand codes (before colon) into a new column
df['M-lig-code'] = df['M-lig'].apply(lambda lst: [item.split(':')[0].strip() for item in lst])


In [18]:
df.head(11)

,target,malaria,target_renamed,malaria_renamed,T-cid,M-cid,T-lig,M-lig,T-gene,M-gene,...,blast_percent_identity,ligand_D,M_D_align_evalue,M_D_rmsd,M_D_align_len,3d_align_evalue,3d_rmsd,3d_align_len,T-lig-code,M-lig-code
0,1C14,3AM3,1C14_renamed.pdb,3AM3_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",['N/A'],['FabI'],...,25.738,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[NAD, TCL]"
1,1C14,1UH5,1C14_renamed.pdb,1UH5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[TCL: TRICLOSAN, NAD: NICOTINAMIDE-ADENINE-DIN...",['N/A'],['FabI'],...,25.316,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[TCL, NAD]"
2,1C14,2OL4,1C14_renamed.pdb,2OL4_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, JPN: ...",['N/A'],['FabI'],...,24.051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[NAD, JPN]"
3,1C14,1VRW,1C14_renamed.pdb,1VRW_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAI: 1,4-DIHYDRONICOTINAMIDE ADENINE DINUCLEO...",['N/A'],['N/A'],...,24.051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]",[NAI]
4,1C14,3AM5,1C14_renamed.pdb,3AM5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",['N/A'],['fabI'],...,25.316,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[NAD, TCL]"
5,1C14,1NHG,1C14_renamed.pdb,1NHG_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",['N/A'],"['N/A', 'N/A']",...,29.078,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[NAD, TCL]"
6,1D7O,3AM3,1D7O_renamed.pdb,3AM3_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",['N/A'],['FabI'],...,47.975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[NAD, TCL]"
7,1D7O,3AM5,1D7O_renamed.pdb,3AM5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...",['N/A'],['fabI'],...,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[NAD, TCL]"
8,1D7O,1UH5,1D7O_renamed.pdb,1UH5_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[TCL: TRICLOSAN, NAD: NICOTINAMIDE-ADENINE-DIN...",['N/A'],['FabI'],...,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[TCL, NAD]"
9,1D7O,2OL4,1D7O_renamed.pdb,2OL4_renamed.pdb,A,A,"[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, TCL: ...","[NAD: NICOTINAMIDE-ADENINE-DINUCLEOTIDE, JPN: ...",['N/A'],['FabI'],...,47.664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[NAD, TCL]","[NAD, JPN]"


In [19]:
df.to_csv('/content/Ultimate_Dataset.csv', index = False)